In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score


In [2]:
df = pd.read_csv("../data/stroke_data.csv")

# convert all column names to lowercase
df.columns = df.columns.str.lower()

print("Columns:", df.columns.tolist())
print(df.head())


Columns: ['sex', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'residence_type', 'avg_glucose_level', 'bmi', 'smoking_status', 'stroke']
   sex  age  hypertension  heart_disease  ever_married  work_type  \
0  1.0   63             0              1             1          4   
1  1.0   42             0              1             1          4   
2  0.0   61             0              0             1          4   
3  1.0   41             1              0             1          3   
4  1.0   85             0              0             1          4   

   residence_type  avg_glucose_level   bmi  smoking_status  stroke  
0               1             228.69  36.6               1       1  
1               0             105.92  32.5               0       1  
2               1             171.23  34.4               1       1  
3               0             174.12  24.0               0       1  
4               1             186.21  29.0               1       1  


In [3]:
X = df.drop("stroke", axis=1)
y = df["stroke"]


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [5]:
# handle missing numeric values
num_imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(num_imputer.fit_transform(X_train), columns=X_train.columns)
X_test = pd.DataFrame(num_imputer.transform(X_test), columns=X_test.columns)


In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [7]:
k_value = min(8, X_train.shape[1])
selector = SelectKBest(score_func=f_classif, k=k_value)

X_train_selected = selector.fit_transform(X_train_scaled, y_train)
X_test_selected = selector.transform(X_test_scaled)


In [8]:
lr_model = LogisticRegression(max_iter=1000, random_state=42)
rf_model = RandomForestClassifier(random_state=42)

lr_model.fit(X_train_selected, y_train)
rf_model.fit(X_train_selected, y_train)


def evaluate_model(model, X, y):
    y_pred = model.predict(X)

    return {
        "Accuracy": accuracy_score(y, y_pred),
        "Precision": precision_score(y, y_pred),
        "Recall": recall_score(y, y_pred),
        "F1": f1_score(y, y_pred)
    }


lr_results = evaluate_model(lr_model, X_test_selected, y_test)
rf_results = evaluate_model(rf_model, X_test_selected, y_test)

print("\nLogistic Regression:", lr_results)
print("Random Forest:", rf_results)


best_model = rf_model if rf_results["F1"] > lr_results["F1"] else lr_model

print("\nBest Model Selected:",
      "Random Forest" if best_model == rf_model else "Logistic Regression")



Logistic Regression: {'Accuracy': 0.6774627230505988, 'Precision': 0.7182209469153515, 'Recall': 0.6016826923076923, 'F1': 0.6548070634401569}
Random Forest: {'Accuracy': 0.9546565631874847, 'Precision': 0.929299796057104, 'Recall': 0.9858173076923077, 'F1': 0.9567246004899101}

Best Model Selected: Random Forest


In [9]:
final_model = best_model

best_threshold = 0.5
best_f1 = 0

y_probs = final_model.predict_proba(X_test_selected)[:, 1]

for thresh in np.arange(0.3, 0.91, 0.05):
    y_pred_thresh = (y_probs >= thresh).astype(int)
    f1 = f1_score(y_test, y_pred_thresh)

    if f1 > best_f1:
        best_f1 = f1
        best_threshold = thresh

print("\nBest Threshold:", best_threshold)
print("Best F1 Score:", best_f1)

y_final_pred = (y_probs >= best_threshold).astype(int)

print("\nFinal Evaluation:")
print("Accuracy:", accuracy_score(y_test, y_final_pred))
print("Precision:", precision_score(y_test, y_final_pred))
print("Recall:", recall_score(y_test, y_final_pred))
print("F1 Score:", f1_score(y_test, y_final_pred))



Best Threshold: 0.5999999999999999
Best F1 Score: 0.9588354611621525

Final Evaluation:
Accuracy: 0.9578342703495478
Precision: 0.9519071310116086
Recall: 0.9658653846153846
F1 Score: 0.9588354611621525


In [10]:
model_data = {
    "model": final_model,
    "imputer": num_imputer,
    "scaler": scaler,
    "selector": selector,
    "threshold": best_threshold,
    "feature_columns": X.columns.tolist()
}

joblib.dump(model_data, "../models/stroke_model.pkl")

print("\nModel saved successfully!")



Model saved successfully!
